# Frontends: Python and R bindings

The Python and R front ends (`python/mboostrs/` and `mboostrs/`) parse formulas
and data frames, convert them to the arrays and structs accepted by the Rust
core, and retain the fitted state in model objects. This chapter describes
formula parsing, model state, core calls, prediction, plotting and serialization. It is the
contributor-facing companion to [](architecture.ipynb). Read
[](architecture.ipynb) first for the crate layout.

Both front-ends call the Rust entry points in `crates/mboostrs-py` (PyO3) and
`crates/mboostrs-r` (extendr). The shared core performs the numerical
calculations for both languages; parity tests compare input conversions and
returned results.

## The two binding crates

| Crate | Mechanism | Lives in | Called from |
|---|---|---|---|
| `mboostrs-py` | PyO3 `#[pyfunction]` | `crates/mboostrs-py/src/lib.rs` | `python/mboostrs/_rust` extension |
| `mboostrs-r` | extendr `extendr_module!` | `crates/mboostrs-r/src/lib.rs` | `mboostrs/` via `.Call("wrap__<name>", ...)` |

Both bindings are **translation layers**: they unpack arrays, build the
learner objects the core expects, call `fit_with` / `fit_lss`, and repackage
the resulting `BoostingPath`. They contain no statistics beyond marshalling.
The [fit call](#the-fit-call) shows the array arguments used for GLMBoost and
the ordered learner specifications used for GAMBoost. The extendr layer
converts R lists and columns to the same core types.

## Formula parsing (Python)

The formula parser is pure Python (`python/mboostrs/formula.py`), regex-based.
Two entry points:

- `parse_formula(formula)` (`formula.py:62`) — the simple `y ~ x1 + x2` form
  used by `glmboost`. Returns `(response_name, [predictor_names])`.
- `parse_gamboost_formula(formula)` (`formula.py:305`) — the rich form used by
  `gamboost`, returning `(response_name, [Bbs/Bols/Bmono/...])` learner
  objects.

### The parenthesis-depth tokenizer

The RHS is split on `+` only at parenthesis depth zero (`_split_formula`,
`formula.py:17-43`). Parenthesis-depth tracking preserves kwargs such as
`boundary.knots = c(0, 10)`: the comma inside `c(...)` is at depth 1, so it
does not split the term. The same logic applies to `[` and
`]`. **Never** replace this with a naive `strsplit(.., ",")`; it breaks on
every multi-element `c(...)` value, as does R's `.parse_bl_args`.

### `_KWARG_RE`: matching `c(...)` as a unit

Per-learner kwargs are parsed by `_KWARG_RE` (`formula.py:82`), a six-arm
regex that tries, in order: double-quoted string, single-quoted string,
`c(...)` literal (matched whole so internal commas are not split), another
`name(...)` call, or a bare scalar. The helper `_kwarg_value` picks the first
non-None group. This mirrors R's `.parse_bl_args`.

### Formula-parseable learners

recognized keywords: `bols`, `bbs`, `bmono`, `btree`, `bmrf`, `brandom`,
`bspatial`, `bns`, `bkernel`, `bconstrained` (`formula.py:69-80`).

Three learners cannot round-trip through a formula string because they take
matrix arguments (`buser` is not formula-parseable at all; `bmrf` and
`bconstrained` parse but emit their matrix argument as `None`):

- **`bmrf`** takes a neighborhood matrix `K`. The parser emits `B(mrf)` with
  `K=None`; the caller must replace it before fitting (`formula.py:333`).
- **`bconstrained`** takes a constraint matrix `C`. Same pattern: emitted with
  `C=None` (`formula.py:359`).
- **`buser`** accepts raw design `X` and penalty `K` arrays. It is **not**
  formula-parseable at all; supply it programmatically via
  `gamboost(..., learners=[Buser(...)])` (`gamboost.py:2042`).

## GAMBoostModel state (`gamboost.py`)

The model class is large because it must support `fit[:k]` (truncation),
`predict(newdata=...)`, `aic`, `varimp`, `confint`, and `partial_effects` —
all of which need the original design data. `__init__` (`gamboost.py:139-238`)
groups private state into four buckets:

**(a) Boosting-path state** (the path itself, returned by Rust):

| Field | Meaning |
|---|---|
| `_fitted` | final fitted values $\hat\eta$ |
| `_risk` | risk array, length $m_{\text{stop}}+1$ |
| `_selected` | per-iteration formula-indexed learner indices |
| `_coefficients` | per-step coefficient vectors |
| `_offset`, `_nu` | scalar offset and step length |

**(b) The learner columns** — `_replay_columns`, one array per variable the
base learners read, plus the response. A fitted model needs these to refit on
a slice, to predict on new data, and to rebuild its learner store after
deserialization.


**(c) Index mappings** between the engine's learner order and the formula's:

| Field | Maps |
|---|---|
| `_rust_idx_to_formula_idx` | engine learner index $\rightarrow$ formula position |
| `_formula_idx_to_rust_idx` | formula position $\rightarrow$ engine learner index |

For every fit except Multinomial these are the identity: learners cross the
boundary as a list of records in formula order, so there is nothing to remap.
They are still consulted because Multinomial takes the positional-array call,
which groups all `bbs` learners first, then all `bmono`, and so on. That
grouping is why the mappings are still used: a wrong remap mislabels
`coef` and `partial_effects` results after the first learner whose order
differs between the lists.

**(d) The LearnerStore** — `_learner_store` (`gamboost.py:238`), the PyO3
opaque object returned by Rust. Used by `aic()` (`gamboost.py:343`) and
`hatvalues()` (`gamboost.py:365`). Pickling drops it (see
[below](#serialization-pickling)).

## GLMBoostModel state (`glmboost.py`)

Much leaner (`glmboost.py:66-102`) because GLMBoost only deals with scalar
linear learners: it stores `_predictor_names` (a flat list) plus
`_centering_means` to refit/predict. No per-learner-type dicts, no
`_rust_idx_to_formula_idx` mapping (only one learner type, so the indices line
up directly). Has `_nuisance_vec` (`glmboost.py:101`) for PropOdds (proportional-odds model) and Multinomial threshold parameters — not present on GAMBoostModel (which uses Kronecker
wrapping for multinomial instead).

The one method GLMBoost has that GAMBoost does not is **`coef_path()`**
(`glmboost.py:506-540`): it returns each predictor's cumulative coefficient as
a 1-D array over iterations $0 \ldots m_{\text{stop}}$. The implementation
builds a `(mstop+1, n_active)` contribution matrix, applies `np.cumsum` along
axis 0, and returns the columns keyed by predictor name. GAMBoost lacks this
because per-iteration coefficients are vectors (one per basis column) — there
is no meaningful scalar "coefficient path" for a smooth term.

## GAMLSS front-end state (`gamlss.py`)

`gamlss.py` provides `glmboostLSS`/`gamboostLSS` and their model classes,
following the same pattern as `glmboost.py`/`gamboost.py` but with a formula-
ordered learner list and a `LearnerStore` **per distribution parameter**. For
`gamboostLSS`, nested learner specifications preserve mixed `Bols`, `Bbs`,
`Bns`, and `Brandom` order
through fit, CV, and stability selection. The `predict(parameter=...)` and
`coef(parameter=...)`
methods dispatch to the named parameter's stored state. See
[](lss_engine.ipynb) for the Rust-side `LssFamily` trait and the cyclic/
non-cyclic engine these front-end classes call into.

(the-fit-call)=
## The fit call

GLMBoost passes a matrix of predictor columns. GAMBoost passes an ordered
list of learner specifications, each containing its type and settings.

**GLMBoost** (`glmboost.py:1002`) is the simple case — a single `(n, p)` numpy
array of stacked columns:

```python
(offset_val, selected, coefs_flat, risk, fitted, nuisance_vec,
 learner_store) = _fit_glmboost(
    x_matrix, y, w, mstop, nu, family_name, family_params,
    predictor_names, offset, risk=risk_str, trace=trace_bool,
)
```


**GAMBoost** crosses the production boundary as one self-describing record
per learner. The records remain in formula order, so fit, resampling, and
prediction share the same learner indices:

```python
(offset_val, selected, coefs_vecs, risk, fitted, nuisance, nuisance_path,
 learner_store, execution_report, residual) = _fit_gamboost_specs(
    learner_specs, y, w, mstop, nu, family_name, family_params,
    offset=offset, multinomial_n_classes=multinomial_n_classes,
    risk=risk_str, trace=trace_bool, component_fraction=subsample,
    subsample_seed=subsample_seed,
)
```


`learner_spec()` builds each record from the parsed learner and training
data. `LearnerBlueprint::prepare` returns both the prepared learner and its
`BasisInfo`, so the Rust `LearnerStore` retains the new-data replay recipe
without a second Python-side metadata accumulator.

### The binomial `{-1,+1}` recoding

Python passes raw `{0,1}` to Rust. The engine asks the selected family's
`encode_response()` method for its internal coding before it computes the
offset or gradient. Fit and resampling therefore use the same conversion:

```rust
let y_encoded = family.encode_response(y);
let y = match y_encoded.as_ref() {
    Some(encoded) => encoded.view(),
    None => y.view(),
};
```


This matches R's factor-to-`{-1,+1}` convention. The inverse link
(`families.py:_inverse_link`) compensates with `sigmoid(2*eta)`. Do not move
the recoding into Python — it must happen after the family name is known to
the binding.

(serialization-pickling)=
## Serialization (pickling)

The `LearnerStore` (PyO3 opaque object) has **no native pickle support**.
Pickling is handled Python-side by `__getstate__` / `__setstate__` that drop
the store entirely:

- **GAMBoostModel**: `__getstate__` (`gamboost.py:1835-1883`) omits
  `_learner_store` and `_data`; `__setstate__` (`gamboost.py:1885-1933`)
  explicitly sets `self._learner_store = None`.
- **GLMBoostModel**: same pattern at `glmboost.py:699-716` and `glmboost.py:718-735`.

A `GAMBoostModel` rebuilds the store on first use through
`prepare_learner_store`, which prepares the same learners from the stored
specifications and training weights without fitting. The result is bit-identical
to the store the fit produced, so prediction, `aic()`, and `hatvalues()` all
work after a round trip. `GLMBoostModel` does not rebuild; its `aic()` and
`hatvalues()` still return `None` after unpickling, and it has to be refitted.

There is no `__reduce__` anywhere in the package.

Pickle is the convenience route, and it is only safe within one environment. For
a model that has to survive a dependency bump, move to another language, or sit
in an archive, use the JSON artifact described next.

## Portable JSON artifacts

Pickling ties a fitted model to a Python version, a package version, and a
compiled extension. `mb.save_model(fit, path)` writes a second format that does
not: schema-versioned JSON, readable by three independent implementations.

`python/mboostrs/serialize.py` writes it. Values carry explicit type tags —
`{"__type__": "ndarray", ...}` rather than positional encoding — so a reader can
reject what it does not understand instead of guessing. Families and base
learners are stored by name plus parameters, the transform recipe and factor
levels travel with the model, and the formula is plainly legible in the file.
Artifacts run 15–55 KiB. `tests/test_serialize.py` asserts the file contains no
pickle opcodes.

Three readers consume it:

| Reader | Entry point | Can load | Can score |
|---|---|---|---|
| Python | `mb.load_model(path)` | All three model classes. | Everything it can fit. |
| R | `load_model(path)` in `mboostrs/R/mboostrs_load.R` | `GLMBoostModel`, `GAMBoostModel`. | Linear `glmboost` models, link and response scale. |
| Rust | `Artifact::load(path)` in `crates/mboostrs-core/src/artifact.rs` | `GLMBoostModel`, `GAMBoostModel`. | Linear `glmboost` models, link and response scale. |

A distributional artifact carries one learner block per distribution
parameter, with `model_class` set to `"LSSModel"` and an `lss_family` type
tag. R and Rust readers reject an unrecognized `model_class` by name, so a
reader that does not support LSS artifacts reports an error instead of
interpreting the artifact as a linear model.

The R and Rust readers load artifacts for inspection, including their formula,
family, coefficients and execution report. They predict from linear models;
requests to score spline, interaction or transformed models return an error
that directs the caller to the Python reader.

The Rust reader exposes the artifact as typed data: `formula()`, `family()`, `coefficients()`, `selected()`, `nu()`,
`offset()`, `centering_means()`, and `execution_report()` decoding into the
engine's own `FitReport`, with `raw_state()` as an escape hatch. `predict()`
aggregates `nu * coefficient` across the selected iterations and then calls
`crate::path::predict_glmboost` — the same function the PyO3 binding calls for a
live model, rather than a reimplementation of it.

Predictions match Python bit for bit. The parity test compares with `assert_eq!` against stored Python values for Gaussian, Poisson,
and Binomial on both scales.

Bit-identical scoring requires `serde_json`'s `float_roundtrip` feature. The
default float parser is not correctly rounded and parsed a fixture coefficient,
`"0.39173632285007604"`, to a bit pattern one ULP from Python's. The
`float_roundtrip` feature is therefore required rather than optional
(`crates/mboostrs-core/Cargo.toml:38-44`) — for a reader whose purpose is
bit-identical scoring, a parse error of one ULP (unit in the last place, the spacing between adjacent floating-point numbers) is a correctness bug.

A Python test locks the artifact field names the R and Rust readers depend on, so
a Python-side rename fails there rather than silently breaking cross-language
scoring.

## scikit-learn adapter (`sklearn.py`)

Four estimators (`sklearn.py:46-51`), all `BaseEstimator` subclasses:

| Class | Loss | Per-feature learner | Engine |
|---|---|---|---|
| `GLMBoostRegressor` | Gaussian (default) | `bols(fj)` | `glmboost` |
| `GAMBoostRegressor` | Gaussian (default) | `bbs(fj, df=...)` | `gamboost` |
| `GLMBoostClassifier` | Binomial | `bols(fj)` | `glmboost` |
| `GAMBoostClassifier` | Binomial | `bbs(fj, df=...)` | `gamboost` |

### Feature-name convention

Module constants `_FEATURE_PREFIX = "f"` and `_RESPONSE_NAME = "y"`
(`sklearn.py:61-62`). When `X` is a pandas/polars DataFrame, real column names
are captured by `_capture_columns` (`sklearn.py:65`); otherwise the adapter
falls back to `f0, f1, ..., f{p-1}` (`sklearn.py:107`). These are stored on
`est.feature_names_in_` and `est._fit_columns_`, and reused at predict time.

For plain NumPy arrays, examples use `f0, f1` because the adapter assigns
those names when no input column names are available.

### Fit / predict flow

`_fit_boost` (`sklearn.py:96`) runs `check_X_y`, builds the data dict via
`_named_data`, resolves or builds a formula via `_resolve_formula`
(`sklearn.py:81`), calls the appropriate engine (`glmboost` or `gamboost`),
and stores the resulting model on `est.model_`. `_predict_response`
(`sklearn.py:145`) calls `model_.predict(newdata=..., type="response")`.
Classifiers add `predict_proba` via `_BinomialBoostClassifier`
(`sklearn.py:221`): `p1 = _predict_response(self, X)`, then
`column_stack([1-p1, p1])`, and `predict` thresholds at $0.5$.

## R frontend (`mboostrs/`)

The R package wraps the same Rust core through [extendr](https://extendr.github.io/) (the Rust-to-R binding library). The extendr binding lives in
**Rust**, at `crates/mboostrs-r/src/lib.rs` — the `extendr_module!` block at
`lib.rs:1886-1902` registers 14 callable functions, and the registration
entry point at `lib.rs:1904-1910` self-registers via
`R_init_mboostrs`. R calls them through extendr's `wrap__<name>` C wrappers,
e.g. `.Call("wrap__fit_glmboost", ..., PACKAGE = "mboostrs")` at
`mboostrs.R:504`.

### R-side layout

``````
mboostrs/
|-- R/
|   |-- mboostrs.R            boost_control, glmboost, predict/cvrisk/cv_weights wrappers
|   |-- mboostrs_gamboost.R   gamboost(), predict.gamboost, cvrisk, stabsel, partial_effects
|   |-- mboostrs_gamlss.R     glmboostLSS(), gamboostLSS(), cvrisk_*LSS, stabsel_*LSS
|   |-- mboostrs_methods.R    S3 methods: residuals, logLik, summary, varimp, coef_path, hatvalues
|   |-- mboostrs_plots.R      plot methods
|   +-- mboostrs_extra.R      stabsel(), make_grid_lss()
|-- src/
|   |-- Makevars              static-links target/release/libmboostrs_r.a via -Wl,-force_load
|   +-- init.c                empty — Rust cdylib self-registers
+-- man/                      139 .Rd help files
``````

### Exported surface

The NAMESPACE (hand-maintained) exports: `glmboost`, `gamboost`,
`glmboostLSS`, `gamboostLSS`, `boost_control`, `cvrisk`,
`cvrisk_glmboostLSS`, `cvrisk_gamboostLSS`, `cv_weights`, `stabsel`,
`stabsel_glmboostLSS`, `stabsel_gamboostLSS`, `stabsel_params`,
`make_grid_lss`, `confint`, generics for `varimp`/`extract`/`coef_path`/
`selected`/`partial_effects`, all 18 plot methods, and all 18 + 11 family
constructors.

### Python-only features

A small set of features are not yet in the R binding — see
[](../appendix/r_parity_matrix) for the full matrix. Everything else —
fitting, prediction (including Multinomial newdata / truncated-mstop),
cvrisk, AIC, hatvalues, all plotting — is at parity with the Python
binding, with two scope differences. R's `cvrisk` for a `gamboost` model
evaluates the out-of-bag risk with unit weights, so it refuses a model
fitted with non-constant weights, where the Python path weights held-out
rows. R's `stabsel` fits its subsamples through `glmboost`, so it takes
plain predictors and refuses base-learner terms, where Python's `stabsel`
runs on a fitted `gamboost` model.

## Contributing to a front-end

Two common contributor tasks:

1. **Expose a new Rust core function to Python.** Add a `#[pyfunction]` in
   `crates/mboostrs-py/src/lib.rs`, register it in the `pymodule!` block,
   then call it from the appropriate Python file via `mboostrs._rust.<name>`.
   The PyO3 signature defines the accepted arguments and return values. Prefer typed or self-describing
   records when several fields describe one statistical object; the wide
   `fit_gamboost` compatibility symbol is deprecated; new code uses the record-based call.

2. **Expose a new Rust core function to R.** Add a `#[extendr_function]` in
   `crates/mboostrs-r/src/lib.rs`, register it in `extendr_module!`, add the
   `.Call("wrap__<name>", ...)` wrapper in `mboostrs/R/`, add the `export()`
   line to `NAMESPACE`, and write the `.Rd` help file. The hand-maintained
   NAMESPACE means you must do the export step explicitly (extendr's
   auto-generation is not used here).

In both cases the underlying Rust function in `mboostrs-core` is unchanged.

## See also

- [](architecture.ipynb) — the crate layout and the `PreparedLearner` / `Family`
  traits the front-ends build objects for.
- [](engine.ipynb) — how `fit_with` uses the prepared learners and response arrays.
- [](testing.ipynb) — how the parity tests exercise both front-ends against R.
- [](../usage/glmboost), [](../usage/gamboost), and
  [](../usage/gamboostlss) — the Python API calls that enter the front-end
  code this chapter describes.